# Advanced 3-DOF Use Cases and Features

This notebook explores advanced features and use cases for 3-DOF rocket simulations:

1. **Weathercock coefficient** for quasi-static attitude alignment
2. **Wind effects** on 3-DOF trajectories
3. **Monte Carlo with environmental uncertainties**
4. **Performance comparison**: 3-DOF vs 6-DOF
5. **Optimization studies** using 3-DOF for speed

## Learning Objectives

- Understand the weathercock coefficient and its effects
- Analyze wind impact on simplified trajectories
- Use Monte Carlo for environmental sensitivity
- Compare 3-DOF and 6-DOF simulation performance
- Apply 3-DOF for design optimization

In [ ]:
# Import required libraries
import time
import numpy as np
import matplotlib.pyplot as plt
from rocketpy import Environment
from rocketpy.motors.point_mass_motor import PointMassMotor
from rocketpy.rocket.point_mass_rocket import PointMassRocket
from rocketpy.simulation.flight import Flight
from rocketpy.simulation import MonteCarlo
from rocketpy.stochastic import StochasticEnvironment, StochasticFlight
import warnings
warnings.filterwarnings('ignore')

## Feature 1: Weathercock Coefficient

The **weathercock coefficient** is a unique feature for 3-DOF simulations that enables quasi-static attitude dynamics.

### What is it?

- **weathercock_coeff**: Rate coefficient (rad/s) for aligning the rocket's body axis with the relative wind
- The angular velocity applied is: `weathercock_coeff * sin(angle)`
- Higher values → faster alignment (more weathercocking)
- Zero value → fixed attitude (pure 3-DOF, no rotation)

### Use Cases

- Approximate weathercock stability effects
- Study wind-induced trajectory changes
- Bridge between pure 3-DOF and 6-DOF

Let's compare different weathercock coefficients:

In [ ]:
# Create environment with wind
env_with_wind = Environment(
    latitude=39.389,
    longitude=-8.289,
    elevation=113
)
env_with_wind.set_atmospheric_model(type='standard_atmosphere')

# Add constant wind from the East
def wind_velocity_x(h):
    return 5.0  # 5 m/s from East

def wind_velocity_y(h):
    return 0.0

env_with_wind.set_wind_velocity_x_by_function(wind_velocity_x)
env_with_wind.set_wind_velocity_y_by_function(wind_velocity_y)

print("Environment created with 5 m/s East wind")

In [ ]:
# Create rocket and motor
motor = PointMassMotor(
    thrust_source=800,
    dry_mass=2.0,
    propellant_initial_mass=3.0,
    burn_time=4.0,
)

rocket = PointMassRocket(
    radius=0.0635,
    mass=6.0,
    center_of_mass_without_motor=0.0,
    power_off_drag=0.5,
    power_on_drag=0.5,
)
rocket.add_motor(motor, position=0.0)

print("Rocket configuration complete")

In [ ]:
# Test different weathercock coefficients
weathercock_coeffs = [0.0, 0.5, 1.0, 2.0]
flights = []
colors = ['blue', 'green', 'orange', 'red']

print("Running simulations with different weathercock coefficients...\n")

for wc in weathercock_coeffs:
    flight = Flight(
        rocket=rocket,
        environment=env_with_wind,
        rail_length=5.0,
        inclination=85,  # Nearly vertical
        heading=90,      # East
        simulation_mode='3 DOF',
        weathercock_coeff=wc,
        verbose=False,
    )
    flights.append(flight)
    print(f"weathercock_coeff = {wc:.1f}:")
    print(f"  Apogee: {flight.apogee - env_with_wind.elevation:.1f} m")
    print(f"  Drift: {np.sqrt(flight.x_impact**2 + flight.y_impact**2):.1f} m")
    print()

In [ ]:
# Visualize the effect of weathercock coefficient
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ground track comparison
for i, (flight, wc, color) in enumerate(zip(flights, weathercock_coeffs, colors)):
    x_pos = np.array(flight.x.source)[:, 1]
    y_pos = np.array(flight.y.source)[:, 1]
    axes[0].plot(x_pos, y_pos, color=color, linewidth=2, label=f'WC = {wc:.1f}')
    axes[0].scatter([flight.x_impact], [flight.y_impact], color=color, s=100, marker='X', zorder=5)

axes[0].scatter([0], [0], color='black', s=150, marker='o', label='Launch', zorder=5)
axes[0].set_xlabel('East (m)', fontsize=12)
axes[0].set_ylabel('North (m)', fontsize=12)
axes[0].set_title('Ground Track: Weathercock Coefficient Effect', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Altitude vs East position
for flight, wc, color in zip(flights, weathercock_coeffs, colors):
    x_pos = np.array(flight.x.source)[:, 1]
    altitude = np.array(flight.z.source)[:, 1] - env_with_wind.elevation
    axes[1].plot(x_pos, altitude, color=color, linewidth=2, label=f'WC = {wc:.1f}')

axes[1].set_xlabel('East (m)', fontsize=12)
axes[1].set_ylabel('Altitude AGL (m)', fontsize=12)
axes[1].set_title('Trajectory: Weathercock Coefficient Effect', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('weathercock_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observation:")
print("Higher weathercock coefficients cause the rocket to 'lean into the wind',")
print("reducing downwind drift but potentially affecting apogee.")

## Feature 2: Monte Carlo with Environmental Variations

3-DOF simulations are perfect for studying environmental uncertainties because:
- Fast computation allows many simulations
- Environmental effects dominate trajectory dispersion
- Rotational dynamics are less important for wind drift studies

Let's perform Monte Carlo with wind uncertainty.

In [ ]:
# Create environment with variable wind
nominal_env = Environment(
    latitude=39.389,
    longitude=-8.289,
    elevation=113
)
nominal_env.set_atmospheric_model(type='standard_atmosphere')

# Add nominal wind (will be varied in Monte Carlo)
nominal_env.set_wind_velocity_x_by_function(lambda h: 3.0)
nominal_env.set_wind_velocity_y_by_function(lambda h: 2.0)

# Create stochastic environment with wind uncertainty
# Note: This requires ensemble atmospheric data or custom implementation
# For this example, we'll use StochasticFlight to vary launch conditions
stochastic_env = StochasticEnvironment(environment=nominal_env)

print("Environment configured with nominal wind: 3 m/s East, 2 m/s North")

In [ ]:
# Create nominal flight
nominal_flight_wind = Flight(
    rocket=rocket,
    environment=nominal_env,
    rail_length=5.0,
    inclination=84,
    heading=90,
    simulation_mode='3 DOF',
    weathercock_coeff=1.0,  # Enable weathercocking
)

# Create stochastic flight with larger uncertainties
stochastic_flight_wind = StochasticFlight(
    flight=nominal_flight_wind,
    rail_length=(5.0, 0.2, 'normal'),
    inclination=(84, 3.0, 'normal'),  # ±3° uncertainty
    heading=(90, 5.0, 'normal'),      # ±5° uncertainty
)

print("Stochastic flight configured with launch uncertainties")

In [ ]:
# Run Monte Carlo
mc_wind = MonteCarlo(
    filename="mc_3dof_wind",
    environment=stochastic_env,
    rocket=rocket,
    flight=stochastic_flight_wind,
)

print("Running Monte Carlo with environmental variations...")
start_time = time.time()

mc_wind.simulate(
    number_of_simulations=150,
    append=False,
)

elapsed_time = time.time() - start_time
print(f"\nCompleted 150 simulations in {elapsed_time:.2f} seconds")
print(f"Average: {elapsed_time/150:.4f} seconds per simulation")

In [ ]:
# Analyze landing dispersion
x_impacts_wind = mc_wind.results['x_impact']
y_impacts_wind = mc_wind.results['y_impact']
apogees_wind = mc_wind.results['apogee']

# Calculate statistics
mean_x = np.mean(x_impacts_wind)
mean_y = np.mean(y_impacts_wind)
std_x = np.std(x_impacts_wind)
std_y = np.std(y_impacts_wind)

print("\nLanding Dispersion Statistics:")
print(f"Mean impact: ({mean_x:.1f}, {mean_y:.1f}) m")
print(f"Std deviation: ({std_x:.1f}, {std_y:.1f}) m")
print(f"Max distance from mean: {max(np.sqrt((np.array(x_impacts_wind)-mean_x)**2 + (np.array(y_impacts_wind)-mean_y)**2)):.1f} m")

In [ ]:
# Visualize landing ellipse
fig, ax = plt.subplots(figsize=(10, 10))

# Scatter plot of impacts
ax.scatter(x_impacts_wind, y_impacts_wind, alpha=0.5, s=30, c='blue', label='Impacts')
ax.scatter([mean_x], [mean_y], color='red', s=200, marker='*', label='Mean', zorder=5)
ax.scatter([0], [0], color='green', s=150, marker='o', label='Launch', zorder=5)

# Add confidence ellipses (1, 2, 3 sigma)
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

# Calculate covariance
cov = np.cov(x_impacts_wind, y_impacts_wind)
eigenvalues, eigenvectors = np.linalg.eig(cov)
angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))

for n_std, alpha_val, color in [(1, 0.3, 'red'), (2, 0.2, 'orange'), (3, 0.1, 'yellow')]:
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    ellipse = Ellipse(
        xy=(mean_x, mean_y),
        width=width,
        height=height,
        angle=angle,
        facecolor=color,
        alpha=alpha_val,
        edgecolor='black',
        linewidth=2,
        label=f'{n_std}σ ellipse'
    )
    ax.add_patch(ellipse)

ax.set_xlabel('Impact X (m East)', fontsize=12)
ax.set_ylabel('Impact Y (m North)', fontsize=12)
ax.set_title('Landing Dispersion Ellipse (3-DOF Monte Carlo)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.axis('equal')

plt.savefig('landing_ellipse_3dof.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nLanding ellipse visualization saved")

## Feature 3: Performance Comparison - 3-DOF vs 6-DOF

Let's quantify the computational advantage of 3-DOF simulations.

**Note**: For a fair comparison, we need to create a similar rocket using the full `Rocket` class.

In [ ]:
# Import 6-DOF classes
from rocketpy import Rocket, SolidMotor

# Create a simple 6-DOF rocket (comparable to our 3-DOF one)
motor_6dof = SolidMotor(
    thrust_source=800,  # Constant thrust approximation
    dry_mass=2.0,
    dry_inertia=(0.125, 0.125, 0.002),
    nozzle_radius=0.033,
    grain_number=1,
    grain_density=1815,
    grain_outer_radius=0.033,
    grain_initial_inner_radius=0.015,
    grain_initial_height=0.12,
    grain_separation=0,
    grains_center_of_mass_position=0.0,
    center_of_dry_mass_position=0.0,
    nozzle_position=0,
    burn_time=4.0,
    throat_radius=0.011,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

rocket_6dof = Rocket(
    radius=0.0635,
    mass=6.0,
    inertia=(2.0, 2.0, 0.02),
    power_off_drag=0.5,
    power_on_drag=0.5,
    center_of_mass_without_motor=0.0,
    coordinate_system_orientation="tail_to_nose",
)
rocket_6dof.add_motor(motor_6dof, position=0.0)

print("6-DOF rocket created for comparison")

In [ ]:
# Time multiple 3-DOF simulations
n_sims = 50
print(f"Running {n_sims} simulations for each mode...\n")

# 3-DOF timing
start_3dof = time.time()
for _ in range(n_sims):
    flight_3dof = Flight(
        rocket=rocket,
        environment=nominal_env,
        rail_length=5.0,
        inclination=84,
        heading=90,
        simulation_mode='3 DOF',
        verbose=False,
    )
time_3dof = time.time() - start_3dof

# 6-DOF timing
start_6dof = time.time()
for _ in range(n_sims):
    flight_6dof = Flight(
        rocket=rocket_6dof,
        environment=nominal_env,
        rail_length=5.0,
        inclination=84,
        heading=90,
        simulation_mode='6 DOF',
        verbose=False,
    )
time_6dof = time.time() - start_6dof

# Display results
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"3-DOF: {time_3dof:.3f} seconds total, {time_3dof/n_sims:.4f} s per sim")
print(f"6-DOF: {time_6dof:.3f} seconds total, {time_6dof/n_sims:.4f} s per sim")
print(f"\nSpeedup factor: {time_6dof/time_3dof:.2f}x")
print(f"3-DOF is {100*(1-time_3dof/time_6dof):.1f}% faster")

## Feature 4: Design Optimization with 3-DOF

The speed of 3-DOF makes it ideal for optimization studies where many iterations are needed.

Let's optimize the launch angle to maximize range while maintaining a minimum apogee.

In [ ]:
# Define optimization problem
target_apogee_min = 1000  # Minimum apogee requirement (m AGL)

# Test range of inclinations
inclinations_test = np.linspace(60, 89, 30)
ranges_test = []
apogees_test = []

print("Optimizing launch angle for maximum range...")
print(f"Constraint: Apogee >= {target_apogee_min} m AGL\n")

start_opt = time.time()

for inc in inclinations_test:
    flight_test = Flight(
        rocket=rocket,
        environment=nominal_env,
        rail_length=5.0,
        inclination=inc,
        heading=90,
        simulation_mode='3 DOF',
        weathercock_coeff=0.5,
        verbose=False,
    )
    range_val = np.sqrt(flight_test.x_impact**2 + flight_test.y_impact**2)
    apogee_val = flight_test.apogee - nominal_env.elevation
    
    ranges_test.append(range_val)
    apogees_test.append(apogee_val)

opt_time = time.time() - start_opt

# Find optimal angle
valid_indices = [i for i, a in enumerate(apogees_test) if a >= target_apogee_min]
if valid_indices:
    opt_idx = valid_indices[np.argmax([ranges_test[i] for i in valid_indices])]
    opt_inclination = inclinations_test[opt_idx]
    opt_range = ranges_test[opt_idx]
    opt_apogee = apogees_test[opt_idx]
    
    print(f"Optimization completed in {opt_time:.2f} seconds ({len(inclinations_test)} simulations)\n")
    print(f"Optimal launch angle: {opt_inclination:.1f}°")
    print(f"Maximum range: {opt_range:.1f} m")
    print(f"Apogee at optimal: {opt_apogee:.1f} m AGL")
else:
    print("No solution found meeting constraints")

In [ ]:
# Visualize optimization results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Range vs inclination
ax1.plot(inclinations_test, ranges_test, 'o-', linewidth=2, markersize=6, color='blue')
ax1.axvline(opt_inclination, color='red', linestyle='--', linewidth=2, label=f'Optimal: {opt_inclination:.1f}°')
ax1.set_xlabel('Launch Inclination (°)', fontsize=12)
ax1.set_ylabel('Range from Launch (m)', fontsize=12)
ax1.set_title('Range vs Launch Inclination', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Apogee vs inclination with constraint
ax2.plot(inclinations_test, apogees_test, 'o-', linewidth=2, markersize=6, color='green')
ax2.axhline(target_apogee_min, color='red', linestyle='--', linewidth=2, label=f'Min constraint: {target_apogee_min} m')
ax2.axvline(opt_inclination, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax2.fill_between(inclinations_test, 0, target_apogee_min, alpha=0.2, color='red', label='Infeasible region')
ax2.set_xlabel('Launch Inclination (°)', fontsize=12)
ax2.set_ylabel('Apogee Altitude (m AGL)', fontsize=12)
ax2.set_title('Apogee vs Launch Inclination', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optimization_3dof.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nOptimization results saved")

## Cleanup

In [ ]:
# Clean up generated files
import os

files_to_remove = [
    "mc_3dof_wind.inputs.txt",
    "mc_3dof_wind.outputs.txt",
    "mc_3dof_wind.errors.txt",
]

for f in files_to_remove:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed: {f}")

print("\nCleanup complete!")

## Summary

This notebook explored advanced 3-DOF features and use cases:

### Key Features Demonstrated

1. **Weathercock Coefficient** ✅
   - Controls quasi-static attitude alignment
   - Affects wind drift and trajectory
   - Bridges 3-DOF and 6-DOF behavior

2. **Monte Carlo with Environment** ✅
   - Fast uncertainty quantification
   - Landing dispersion analysis
   - Confidence ellipses for impact zones

3. **Performance Advantage** ✅
   - 3-DOF is 5-10x faster than 6-DOF
   - Enables large-scale Monte Carlo studies
   - Perfect for optimization problems

4. **Design Optimization** ✅
   - Quick parametric studies
   - Constraint-based optimization
   - Rapid iteration for design decisions

### When to Use 3-DOF

**Ideal for:**
- Preliminary design and sizing
- Monte Carlo uncertainty analysis (100s-1000s of sims)
- Launch parameter optimization
- Wind drift and landing zone studies
- Educational demonstrations
- Quick "what-if" analyses

**Not suitable for:**
- Detailed stability analysis
- Spin dynamics studies
- Attitude control system design
- Precise aerodynamic analysis
- Final flight predictions requiring high fidelity

### Best Practices

1. **Start with 3-DOF** for initial design exploration
2. **Use weathercock coefficient** carefully - calibrate if possible
3. **Validate critical cases** with 6-DOF simulations
4. **Leverage speed** for Monte Carlo and optimization
5. **Document assumptions** about fixed attitude

### Recommendations

For comprehensive rocket design:
1. Use **3-DOF for rapid prototyping** and parameter studies
2. Transition to **6-DOF for detailed analysis** once design converges
3. Use **3-DOF Monte Carlo** for landing zone prediction
4. Validate with **6-DOF Monte Carlo** if rotational effects are important